In [1]:
import pandas as pd

raw = pd.read_csv("club_signups.csv")
df = raw.copy()

In [2]:
print("shape:", df.shape)   # (rows, columns)
df.dtypes                   # the type of each column

shape: (39, 9)


signup_id       str
student_id      str
full_name       str
email           str
faculty         str
city            str
club            str
signed_up_at    str
fee_paid        str
dtype: object

In [3]:
for col in ["faculty", "city", "club", "fee_paid"]:
    print(f"--- {col}: {df[col].nunique()} distinct values ---")
    print(df[col].value_counts(), "\n")

--- faculty: 13 distinct values ---
faculty
Pharmacy                                  7
EMS                                       7
IET                                       5
Management                                4
Information Engineering and Technology    4
MET                                       3
iet                                       3
Engineering and Materials Science         1
Pharma                                    1
met                                       1
Media Engineering and Technology          1
pharmacy                                  1
ems                                       1
Name: count, dtype: int64 

--- city: 9 distinct values ---
city
Cairo         13
Alexandria    11
Giza           4
Alex           4
cairo          2
giza           2
alexandria     1
CAIRO          1
El Giza        1
Name: count, dtype: int64 

--- club: 13 distinct values ---
club
Robotics       8
Music          5
Chess          4
football       4
Debating       3
Chess Club     

In [4]:
slash = df["signed_up_at"].str.contains("/")              # True for rows in the dd/mm or mm/dd format
parts = df.loc[slash, "signed_up_at"].str.split(" ").str[0].str.split("/")
first  = parts.str[0].astype(int)
second = parts.str[1].astype(int)
print("rows with '/':", slash.sum())
print("first part  ranges", first.min(), "to", first.max())
print("second part ranges", second.min(), "to", second.max())

rows with '/': 14
first part  ranges 13 to 18
second part ranges 9 to 9


In [5]:
iso = pd.to_datetime(df["signed_up_at"], format="%Y-%m-%d %H:%M", errors="coerce")
dmy = pd.to_datetime(df["signed_up_at"], format="%d/%m/%Y %H:%M", errors="coerce")
df["signed_up_at"] = iso.fillna(dmy)

assert df["signed_up_at"].notna().all(), "some dates didn't parse"
df["signed_up_at"].dtype

print("earliest:", df["signed_up_at"].min())
print("latest:  ", df["signed_up_at"].max())

earliest: 2026-09-13 14:36:00
latest:   2026-09-18 23:36:00


In [6]:
fee_map = {"yes": True, "y": True, "true": True, "1": True,
           "no": False, "n": False, "false": False, "0": False}

df["fee_paid"] = df["fee_paid"].str.strip().str.lower().map(fee_map)

assert df["fee_paid"].notna().all(), "unmapped fee_paid value"
df["fee_paid"] = df["fee_paid"].astype(bool)
df["fee_paid"].value_counts()

fee_paid
True     27
False    12
Name: count, dtype: int64

In [7]:
def normalise(s):
    # strip ends, lower-case, collapse internal runs of spaces into one
    return s.str.strip().str.lower().str.replace(r"\s+", " ", regex=True)

faculty_map = {
    "met": "MET", "media engineering and technology": "MET",
    "iet": "IET", "information engineering and technology": "IET",
    "ems": "EMS", "engineering and materials science": "EMS",
    "pharmacy": "Pharmacy", "pharma": "Pharmacy",
    "management": "Management",
}
club_map = {
    "robotics": "Robotics",
    "debate": "Debate", "debating": "Debate", "debate club": "Debate",
    "music": "Music",
    "football": "Football", "soccer": "Football",
    "chess": "Chess", "chess club": "Chess",
}
city_map = {
    "cairo": "Cairo",
    "giza": "Giza", "el giza": "Giza",
    "alexandria": "Alexandria", "alex": "Alexandria",
}

allowed = {
    "faculty": {"MET", "IET", "EMS", "Pharmacy", "Management"},
    "club":    {"Robotics", "Debate", "Music", "Football", "Chess"},
    "city":    {"Cairo", "Giza", "Alexandria"},
}

for col, mapping in [("faculty", faculty_map), ("club", club_map), ("city", city_map)]:
    before = df[col].nunique()
    df[col] = normalise(df[col]).map(mapping)
    assert df[col].notna().all(), f"{col}: a variant is missing from the dictionary"
    assert set(df[col]) <= allowed[col], f"{col}: non-canonical value left"
    print(f"{col}: {before} variants -> {df[col].nunique()}  {sorted(df[col].unique())}")

faculty: 13 variants -> 5  ['EMS', 'IET', 'MET', 'Management', 'Pharmacy']
club: 13 variants -> 5  ['Chess', 'Debate', 'Football', 'Music', 'Robotics']
city: 9 variants -> 3  ['Alexandria', 'Cairo', 'Giza']


In [8]:
df["full_name"] = (df["full_name"]
                   .str.strip()                              # remove spaces at both ends
                   .str.replace(r"\s+", " ", regex=True)     # collapse "  " inside into " "
                   .str.title())                             # "MOHAMED ADEL" -> "Mohamed Adel"

df["email"] = df["email"].str.strip().str.lower()

# checks
assert not df["full_name"].str.contains(r"^\s|\s$|\s{2,}", regex=True).any(), "extra spaces left in a name"
assert (df.groupby("student_id")["full_name"].nunique() == 1).all(), "a student has 2 spellings of their name"
assert (df.groupby("student_id")["email"].nunique() == 1).all(), "a student has 2 spellings of their email"

df.groupby("student_id")[["full_name", "email"]].first()

,full_name,email
student_id,,
55-2639,Youssef Fathy,youssef.fathy@student.guc.edu.eg
55-2971,Salma Ibrahim,salma.ibrahim@student.guc.edu.eg
55-2992,Mohamed Adel,mohamed.adel@student.guc.edu.eg
55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg
58-1089,Karim Kamel,karim.kamel@student.guc.edu.eg
58-1125,Ziad Saleh,ziad.saleh@student.guc.edu.eg
58-8701,Farida Ibrahim,farida.ibrahim@student.guc.edu.eg
58-9118,Mariam Kamel,mariam.kamel@student.guc.edu.eg
61-2495,Ahmed Adel,ahmed.adel@student.guc.edu.eg


In [9]:
n_start = len(df)
n_exact = df.duplicated().sum()
print("rows at start:", n_start)
print("exact duplicate rows:", n_exact)
display(df[df.duplicated(keep=False)].sort_values("signup_id"))   # show them, both copies

df = df.drop_duplicates()
print("rows after removing exact duplicates:", len(df))

rows at start: 39
exact duplicate rows: 3


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Football,2026-09-15 13:39:00,False
34,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Football,2026-09-15 13:39:00,False
15,S022,61-8224,Sara Zaki,sara.zaki@student.guc.edu.eg,EMS,Giza,Music,2026-09-16 01:39:00,True
24,S022,61-8224,Sara Zaki,sara.zaki@student.guc.edu.eg,EMS,Giza,Music,2026-09-16 01:39:00,True
20,S023,58-1125,Ziad Saleh,ziad.saleh@student.guc.edu.eg,IET,Cairo,Chess,2026-09-16 09:29:00,False
33,S023,58-1125,Ziad Saleh,ziad.saleh@student.guc.edu.eg,IET,Cairo,Chess,2026-09-16 09:29:00,False


rows after removing exact duplicates: 36


In [10]:
df = df.sort_values("signed_up_at")                       # oldest first, newest last
dup_mask = df.duplicated(subset=["student_id", "club"], keep="last")
print("older re-submissions removed:", dup_mask.sum())
display(df[df.duplicated(subset=["student_id", "club"], keep=False)]
        .sort_values(["student_id", "club", "signed_up_at"]))  # show the pairs

df = df.drop_duplicates(subset=["student_id", "club"], keep="last")

assert not df.duplicated(subset=["student_id", "club"]).any(), "student_id + club is not unique"
print("final rows:", len(df))

older re-submissions removed: 4


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
8,S020,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Music,2026-09-15 23:07:00,False
9,S033,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Music,2026-09-16 09:07:00,True
22,S021,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Debate,2026-09-16 01:15:00,False
3,S035,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Debate,2026-09-17 11:15:00,True
35,S017,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Debate,2026-09-15 18:01:00,False
25,S034,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Debate,2026-09-16 02:01:00,True
21,S032,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Football,2026-09-17 23:36:00,False
38,S036,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Football,2026-09-18 23:36:00,True


final rows: 32


In [11]:
# Same B2 check, but on the RAW data (club spellings NOT fixed yet)
raw_dedup = raw.drop_duplicates()
print("same student+club found BEFORE fixing spellings:",
      raw_dedup.duplicated(subset=["student_id", "club"]).sum())

same student+club found BEFORE fixing spellings: 2


In [12]:
df = df.sort_values("signup_id")                 # tidy order for the output file
df.to_csv("club_signups_clean.csv", index=False)
print(df.dtypes)
print("saved", len(df), "rows")

signup_id                  str
student_id                 str
full_name                  str
email                      str
faculty                    str
city                       str
club                       str
signed_up_at    datetime64[us]
fee_paid                  bool
dtype: object
saved 32 rows
